<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_Layer23_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# HuggingFace authentication
# !pip install -q huggingface_hub
from huggingface_hub import login
from google.colab import userdata
import os

login(userdata.get('HF_TOKEN'))

# Gemma-Titans Phase 2: LM Fine-Tuning

Фаза 2 обучения: переключаемся с послойной дистилляции на language modeling loss.

**Отличия от Phase 1:**
- Выходы студента передаются между слоями (нет `stop_gradient`, нет teacher chain)
- Loss: cross-entropy по токенам вместо per-layer MSE
- `training_phase=2` в конфиге модели
- Веса загружаются из `saved_titans_delta` (результат Phase 1)
- Сниженные LR (веса уже предобучены)

In [ ]:
# 0. Environment Setup
!pip install -q --upgrade "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog
!pip install -q flax==0.12.5 optax==0.2.6 typeguard==4.4.1 seqio ml_dtypes
!pip install importlib_resources

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git

## Start

In [ ]:
import os
# os._exit(0)

In [ ]:
import sys
import os
sys.path.append(os.getcwd())

import jax
import jax.numpy as jnp
import optax
import dataclasses
import numpy as np
import os
import orbax.checkpoint as ocp
import shutil

from gemma import gm

# Our custom Titans integration
import importlib

%cd Titans_jax
# import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
import titans_tree_utils
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    load_all_phase1_layers,
    reconstruct_opt_params, schedule,
)


print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
if jax.default_backend() == "cpu":
    print("\n❌ ERROR: JAX is running on CPU. Training will be 100x slower.")
    print("Please check Runtime -> Change runtime type and select TPU.")
    # raise RuntimeError("TPU not found")

## 2. Гиперпараметры

In [ ]:
from typing import Iterator, Optional


batch_size = 4
max_length = 1024
total_steps = 10000
TARGET_LAYER = 23                # Earlier layers revert to standard Gemma blocks.
                                 # 11 → layers 11,17,23 active (~70GB compile RAM, batch_size=2)
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

# ═══════════════════════════════════════════════════════════
# FIRST_RUN = True  → загрузить Phase 1 чекпойнт из HF
# FIRST_RUN = False → загрузить Phase 2 чекпойнт из HF
# ═══════════════════════════════════════════════════════════
FIRST_RUN = True


_all_titans_layers = ( 23)
print(f"Active Titans layers: {TARGET_LAYER}")


## Configs

In [ ]:
experimental_config = {
    # ═══ Архитектура (ТАКАЯ ЖЕ) ═══
    'heads': 8,
    'dim_head': 128,
    'chunk_size': 32,
    'mlp_depth': 4,

    # ═══ Phase 2 специфика ═══
    'max_grad_norm': 0.5,             # Жёстче clip — веса уже обучены
    'elastic_net_lambda': 0.005,      # Мягкий L1 — предотвращаем разрастание весов
    'huber_loss_delta': 0.1,          # Умеренный huber — робастнее чем MSE
    'diff_view': False,
    'is_look_ahead': False,
    'adaptive_max_lr': 5e-4,  # scaled by every_k_schedule at injection below
}

from optax._src import base
import dataclasses
import optax

WARMUP = 500

b1_schedule = optax.linear_schedule(
    init_value=0.7,        # Start: мало momentum → модель «чувствует» каждый градиент
    end_value=0.90,        # End: больше momentum → сглаженная оптимизация
    transition_steps=2000, # За первые 2000 шагов нарастить
    transition_begin=WARMUP
)

opt_params = {
    "lr_muon": optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=1e-5,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=5e-6
	),
	"beta": 0.90,
	"lr_adam": optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=5e-5,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=5e-6,
	),
	"adam_b1": b1_schedule,
    "adam_b2": 0.85,
    "lr_gate": optax.warmup_cosine_decay_schedule(
	    init_value=5e-4,
	    peak_value=5e-4,
	    warmup_steps=WARMUP,
	    decay_steps=total_steps - WARMUP,
	    end_value=5e-4,
	),
    "gate_b1": b1_schedule,
    "gate_b2": 0.95,
    "every_k_schedule": 4
}

def build_titans_block_config(
    target_layer: int = TARGET_LAYER,
    experimental_config: dict = None,
    every_k_schedule: int = 4,
):
    """Build neural_mem_kwargs and config from experimental_config."""
    from gemma_titans import Gemma3_1B_Titans

    # ec = experimental_config or {}

    # experimental_config = {
    #     'heads': ec.get('heads', 8),
    #     'dim_head': ec.get('dim_head', 128),
    #     'chunk_size': ec.get('chunk_size', 32),
    #     'mlp_depth': ec.get('mlp_depth', 6),
    #     'max_grad_norm': ec.get('max_grad_norm', 0.5),
    #     'elastic_net_lambda': ec.get('elastic_net_lambda', 0.01),
    #     'diff_view': ec.get('diff_view', False),
    #     'is_look_ahead': ec.get('is_look_ahead', False),
    #     'huber_loss_delta': ec.get('huber_loss_delta', None),
    #     'adaptive_max_lr': ec.get('adaptive_max_lr', 1e-4),
    #     'every_k_schedule': every_k_schedule,
    # }

    config = dataclasses.replace(
        Gemma3_1B_Titans.config,
        training_phase=3,
        titans_layer_indices=[TARGET_LAYER],
        titans_first_layer=TARGET_LAYER,
        neural_mem_kwargs=experimental_config,

    )

    return config

def _build_block_kwargs(config, layer_idx):
    """Build Flax Block kwargs from TransformerConfig for a given layer."""
    from gemma.gm.nn import _modules
    attn_type = config.attention_types[layer_idx]
    is_local = attn_type == _modules.AttentionType.LOCAL_SLIDING
    return dict(
        num_heads=config.num_heads,
        num_kv_heads=config.num_kv_heads,
        embed_dim=config.embed_dim,
        head_dim=config.head_dim,
        hidden_dim=config.hidden_dim,
        sliding_window_size=config.sliding_window_size,
        use_post_attn_norm=config.use_post_attn_norm,
        use_post_ffw_norm=config.use_post_ffw_norm,
        attn_logits_soft_cap=config.attn_logits_soft_cap,
        attn_type=attn_type,
        query_pre_attn_scalar=config.query_pre_attn_scalar(),
        transpose_gating_einsum=config.transpose_gating_einsum,
        use_qk_norm=config.use_qk_norm,
        rope_base_frequency=config.local_base_frequency if is_local else config.global_base_frequency,
        rope_scale_factor=config.local_scale_factor if is_local else config.global_scale_factor,
    )


## 1. Загрузка весов


In [ ]:
import hf_checkpoint
importlib.reload(hf_checkpoint)
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    reconstruct_opt_params, schedule,
    load_all_phase1_layers,
)

In [ ]:
HF_CKPT_REPO = "veriga/titans-checkpoints"

def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

merged_params = None
workdir = os.path.abspath(f'./titans_workdir_phase2_from{TARGET_LAYER}')
workdir_checkpoints = os.path.join(workdir, "checkpoints")

if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"📁 Найдена директория {workdir_checkpoints}. Пропускаем загрузку весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
else:
    # Определяем, какую фазу загружать
    load_phase = 1 if FIRST_RUN else 2
    phase_label = f"Phase {load_phase}"

    # ── Авто-определение последнего чекпойнта ──
    last_meta = load_last_metadata(
        repo_id=HF_CKPT_REPO,
        phase=load_phase,
        # token=userdata.get('HF_TOKEN'),
    )

    if last_meta is not None:
        # Восстанавливаем experimental_config
        experimental_config = last_meta.get("experimental_config", experimental_config)
        print(f"📋 Restored experimental_config: {experimental_config}")

        # Восстанавливаем opt_params: schedules → callable
        if "opt_params" in last_meta:
            opt_params = reconstruct_opt_params(last_meta["opt_params"])
            print(f"📋 Restored opt_params with schedules: {list(opt_params.keys())}")

        # Восстанавливаем warm_up
        if "warm_up" in last_meta:
            WARMUP = last_meta["warm_up"]
            print(f"📋 Restored warm_up: {WARMUP}")
    else:
        print("⚠️ Last metadata not found — using current notebook values")
    active_titans_layers = (TARGET_LAYER,)
    active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
    if FIRST_RUN:
        # ── Загружаем веса для ВСЕХ обученных слоёв из Phase 1 ──
        loaded_titans_params = load_all_phase1_layers(
            repo_id=HF_CKPT_REPO,
            titans_first_layer=TARGET_LAYER,
            local_dir=".",
            # token=userdata.get('HF_TOKEN'),
        )


        if loaded_titans_params is not None:
            # Keep only weights for active Titans layers

            loaded_titans_params = {
                k: v for k, v in loaded_titans_params.items()
                if k in active_layer_keys
            }
            print(f"Merging Titans weights for: {sorted(loaded_titans_params.keys())}")

            print("Loading Gemma base weights...")

            original_params = gm.ckpts.load_params(
                gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
                )

            merged_params = titans_tree_utils.extract_and_merge_frozen_head(
                gemma_params=original_params,
                titans_params=loaded_titans_params,
                after_layer=TARGET_LAYER,
                titans_layer_indices = [TARGET_LAYER],
                remove_dead_attn=True
            )
            print(f"✅ Phase 1 weights loaded from HF and merged.")
        else:
            print("⚠️ Не найдено обученных слоёв Phase 1 на HF!")
    else:
        # ── Загружаем комбинированный чекпойнт Phase 2 ──
        if last_meta is not None:
            TARGET_LAYER = last_meta.get("first_layer", TARGET_LAYER)
            total_steps = last_meta.get("total_steps", total_steps)
            print(f"📋 Checkpoint: {last_meta.get('checkpoint')}")

        ckpt_dir = load_checkpoint_from_hf(
            repo_id=HF_CKPT_REPO,
            phase=2,
            first_layer=TARGET_LAYER,
            titans_layer_indices = (TARGET_LAYER,),
            total_steps=total_steps,
            local_dir=".",
        )

        if ckpt_dir is not None:
            print("Loading Gemma base weights...")
            original_params = gm.ckpts.load_params("C:\\Users\\LiveComp\\Titans\\gemma3_1b_ckpt\\gemma3-1b-it")

            print(f"Loading Phase 2 Titans weights from HF...")
            loaded_titans_params = load_titans_weights(ckpt_dir)

            # Keep only weights for active Titans layers
            loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}

            print(f"Merging Titans weights for: {active_layer_keys}")

            merged_params = titans_tree_utils.merge_titans_params(
                original_params, loaded_titans_params, remove_dead_attn=True
            )
            print(f"✅ Phase 2 weights loaded from HF and merged.")
        else:
            print(f"⚠️ Чекпойнт Phase 2 не найден на HF!")


## 4. Датасет подготовленных активаций

In [ ]:

class HFActivationLoader:
    """
    Streams precomputed activation shards from HuggingFace Hub or local disk.

    When ``local_activation_dir`` is provided, reads .npy shards directly
    from the local directory — no HF download, maximum speed.

    When ``local_activation_dir`` is None, loads .npy shards from a folder
    inside an HF dataset repository
    (e.g. ``veriga/openwebtext-gemma3-tokenized-1024/activations_layer23``)
    and pairs them with the corresponding original token IDs from the same
    repo for next-token CrossEntropy targets.

    Features:
      - **Local mode**: set ``local_activation_dir`` to skip HF entirely.
      - Streaming: shards are loaded on-demand (no full load into RAM).
      - Token pairing: each activation batch is paired with original tokens
        from the token dataset for CE loss computation.
      - Shuffle buffer: example-level shuffling via an in-memory buffer.

    Usage (remote)::

        loader = HFActivationLoader(
            activation_repo="veriga/openwebtext-gemma3-tokenized-1024",
            activation_folder="activations_layer23",
            batch_size=4,
        )

    Usage (local — fast, no HF download)::

        loader = HFActivationLoader(
            activation_repo="veriga/openwebtext-gemma3-tokenized-1024",
            activation_folder="activations_layer23",
            local_activation_dir="./activations_layer23",
            batch_size=4,
        )

        for batch in loader:
            hidden = batch["hidden"]   # (B, L, 1152)
            tokens = batch["tokens"]   # (B, L)
            mask   = batch["mask"]     # (B, L)
    """

    def __init__(
        self,
        activation_repo: str,
        activation_folder: str = "",
        local_activation_dir: Optional[str] = None,
        batch_size: int = 4,
        seq_len: int = 1024,
        shuffle: bool = True,
        repeat: bool = True,
        seed: int = 42,
        buffer_size: int = 1000,
        hf_token: Optional[str] = None,
        cache_dir: Optional[str] = None,
    ):
        """
        Args:
            activation_repo: HuggingFace dataset repo containing activation
                shards and tokens (e.g.
                ``"veriga/openwebtext-gemma3-tokenized-1024"``).
            activation_folder: Folder inside the repo where .npy shards and
                metadata.json live (default: ``"activations_layer23"``).
            local_activation_dir: **If set**, read shards and metadata directly
                from this local directory — no HF download needed. This is
                ideal for fast testing when you already ran
                ``precompute_activations.py`` locally. When set, only tokens
                come from HF (for CE loss targets).
            batch_size: Number of examples per training batch.
            seq_len: Sequence length — must match precomputed activations.
            shuffle: Whether to shuffle examples across shards.
            seed: Random seed for shuffling.
            buffer_size: Shuffle-buffer size (in examples).
            hf_token: HuggingFace API token (falls back to ``$HF_TOKEN``).
            cache_dir: Local directory for caching HF downloads (tokens only
                when using local_activation_dir).
        """
        self.activation_repo = activation_repo
        self.activation_folder = activation_folder
        self.local_activation_dir = local_activation_dir
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.shuffle = shuffle
        self.repeat = repeat
        self.seed = seed
        self.buffer_size = buffer_size
        self.hf_token = hf_token or os.environ.get("HF_TOKEN")
        self.cache_dir = cache_dir

        # --- Discover shards ---
        if local_activation_dir:
            # Local mode: read directly from disk
            from glob import glob

            # Only match shard_XXXXXX.npy, NOT shard_XXXXXX_tokens.npy or _masks.npy
            import re
            all_npy = glob(os.path.join(local_activation_dir, "shard_*.npy"))
            self.shard_paths_local = sorted(
                p for p in all_npy
                if re.match(r"shard_\d+\.npy$", os.path.basename(p))
            )
            if not self.shard_paths_local:
                raise FileNotFoundError(
                    f"No shard_*.npy files in {local_activation_dir}"
                )
            self.shard_names = [os.path.basename(p) for p in self.shard_paths_local]
            self._local_mode = True

            # Load metadata from local file
            meta_path = os.path.join(local_activation_dir, "metadata.json")
            if not os.path.exists(meta_path):
                raise FileNotFoundError(f"No metadata.json in {local_activation_dir}")
            with open(meta_path) as f:
                self.metadata = json.load(f)

            print(
                f"📦 HFActivationLoader [LOCAL]: {len(self.shard_names)} shards "
                f"from {local_activation_dir}"
            )
        else:
            # Remote mode: discover & download from HuggingFace
            from huggingface_hub import HfApi

            api = HfApi(token=self.hf_token)
            all_files = api.list_repo_files(
                repo_id=activation_repo,
                repo_type="dataset",
            )

            import re
            if activation_folder:
                prefix = activation_folder + "/"
            else:
                prefix = ""  # files at repo root

            all_npy = [
                p for p in all_files
                if p.startswith(prefix) and p.endswith(".npy")
                and ("/" not in p[len(prefix):])  # no subfolders
            ]
            # ALL .npy files in the folder (for token/mask lookup)
            self._all_npy_names = set(p[len(prefix):] for p in all_npy)
            # Only activation shards (shard_XXXXXX.npy), not _tokens/_masks
            shard_paths = sorted(
                p for p in all_npy
                if re.match(r"shard_\d+\.npy$", p[len(prefix):])
            )
            self.shard_names = [p[len(prefix):] for p in shard_paths]

            if not self.shard_names:
                matching = [f for f in all_files if f.startswith(prefix)]
                raise FileNotFoundError(
                    f"No .npy shards in {activation_repo}"
                    f"/{activation_folder}. "
                    f"Files there: {matching[:10]}"
                )

            self._local_mode = False
            self.shard_paths_local = None

            # Download metadata from HF
            self.metadata = self._load_metadata()

            print(
                f"📦 HFActivationLoader [HF]: {len(self.shard_names)} shards "
                f"from {activation_repo}/{activation_folder}"
            )

        self.embed_dim = self.metadata.get("embed_dim", 1152)
        self.total_examples = self.metadata.get("total_examples", 0)
        self._shard_batch_size = self.metadata.get("batch_size", 8)

        print(
            f"   ~{self.total_examples:,} examples, embed_dim={self.embed_dim}"
        )

        # --- Token dataset (lazy) ---
        self._token_ds = None

    # ---- internal helpers ------------------------------------------------

    def _hf_path(self, filename: str) -> str:
        """Full repo-internal path for a file in the activation folder."""
        if self.activation_folder:
            return f"{self.activation_folder}/{filename}"
        return filename

    def _load_metadata(self) -> dict:
        """Download metadata.json from the HF repo."""
        from huggingface_hub import hf_hub_download

        local_path = hf_hub_download(
            repo_id=self.activation_repo,
            filename=self._hf_path("metadata.json"),
            repo_type="dataset",
            token=self.hf_token,
            cache_dir=self.cache_dir,
        )
        with open(local_path) as f:
            return json.load(f)

    def _download_shard(self, shard_name: str) -> str:
        """Download a single .npy shard, return the local path."""
        from huggingface_hub import hf_hub_download

        return hf_hub_download(
            repo_id=self.activation_repo,
            filename=self._hf_path(shard_name),
            repo_type="dataset",
            token=self.hf_token,
            cache_dir=self.cache_dir,
        )

    def _download_shard_bytes(self, shard_name: str) -> bytes:
        """Download a shard into memory (no disk cache for this call)."""
        from huggingface_hub import hf_hub_download
        local = hf_hub_download(
            repo_id=self.activation_repo,
            filename=self._hf_path(shard_name),
            repo_type="dataset",
            token=self.hf_token,
            cache_dir=self.cache_dir,
        )
        with open(local, "rb") as f:
            return f.read()

    def _load_token_dataset(self):
        """Lazy-load the token dataset from the same HF repo."""
        if self._token_ds is not None:
            return self._token_ds

        from datasets import load_dataset

        self._token_ds = load_dataset(
            self.activation_repo,
            split="train",
            token=self.hf_token,
            cache_dir=self.cache_dir,
        )
        print(f"📄 Token dataset loaded: {len(self._token_ds):,} examples")
        return self._token_ds

    # ---- iteration -------------------------------------------------------

    def _iter_shards(self) -> Iterator[dict]:
        """
        Yield shards as dicts {hidden, tokens, masks}, optionally shuffled.

        In local mode: reads shard_XXXXXX.npy, shard_XXXXXX_tokens.npy,
        shard_XXXXXX_masks.npy from disk.
        In remote mode: downloads .npy from HF; falls back to HF token
        dataset for tokens if _tokens.npy files are not available.
        """
        rng = np.random.default_rng(self.seed)
        indices = list(range(len(self.shard_names)))

        if self.shuffle:
            rng.shuffle(indices)

        for idx in indices:
            if self._local_mode:
                base_path = self.shard_paths_local[idx]
                hidden = np.load(base_path)

                # Derive token/mask paths from activation shard name
                dir_path = os.path.dirname(base_path)
                base_name = os.path.splitext(os.path.basename(base_path))[0]
                tokens_path = os.path.join(dir_path, f"{base_name}_tokens.npy")
                masks_path = os.path.join(dir_path, f"{base_name}_masks.npy")

                if os.path.exists(tokens_path):
                    tokens = np.load(tokens_path)
                else:
                    tokens = None

                if os.path.exists(masks_path):
                    masks = np.load(masks_path)
                else:
                    # Fall back: derive mask from hidden (nonzero rows)
                    masks = (np.abs(hidden).sum(axis=-1) > 1e-8).astype(np.int32)

                yield {"hidden": hidden, "tokens": tokens, "masks": masks}

            else:
                name = self.shard_names[idx]
                local_path = self._download_shard(name)
                hidden = np.load(local_path)
                # np.load returns VoidDType for bfloat16 (numpy doesn't support bf16 natively)
                if not np.issubdtype(hidden.dtype, np.number):
                    import ml_dtypes
                    hidden = hidden.view(ml_dtypes.bfloat16)

                # Try to download co-located tokens/masks
                base = os.path.splitext(name)[0]
                tokens_name = f"{base}_tokens.npy"
                masks_name = f"{base}_masks.npy"

                tokens = None
                # Safe mask creation: handle VoidDType or other non-numeric arrays
                try:
                    masks = (np.abs(hidden).sum(axis=-1) > 1e-8).astype(np.int32)
                except (TypeError, ValueError):
                    masks = np.ones(hidden.shape[:2], dtype=np.int32)

                # Check if _tokens.npy exists in repo
                if tokens_name in self._all_npy_names:
                    try:
                        t_path = self._download_shard(tokens_name)
                        tokens = np.load(t_path)
                        m_path = self._download_shard(masks_name)
                        masks = np.load(m_path)
                    except Exception:
                        pass

                yield {"hidden": hidden, "tokens": tokens, "masks": masks}

    def _iter_examples(self) -> Iterator[dict]:
        """Yield individual examples across all shards."""
        for shard_data in self._iter_shards():
            hidden_shard = shard_data["hidden"]
            tokens_shard = shard_data["tokens"]
            masks_shard = shard_data["masks"]

            for i in range(hidden_shard.shape[0]):
                result = {
                    "hidden": hidden_shard[i],
                    "mask": masks_shard[i],
                }
                if tokens_shard is not None:
                    result["tokens"] = tokens_shard[i]

                yield result

    def _shuffle_buffer(self, it: Iterator[dict]) -> Iterator[dict]:
        """Example-level shuffle buffer."""
        rng = np.random.default_rng(self.seed)
        buf: list = []

        for ex in it:
            buf.append(ex)
            if len(buf) >= self.buffer_size:
                yield buf.pop(rng.integers(0, len(buf)))

        while buf:
            yield buf.pop(rng.integers(0, len(buf)))

    def __iter__(self) -> Iterator[dict]:
        """Yield batches ``{hidden, tokens, mask}`` ready for training."""
        examples = self._iter_examples()

        if self.shuffle:
            examples = self._shuffle_buffer(examples)

        batch: list = []
        for ex in examples:
            batch.append(ex)
            if len(batch) == self.batch_size:
                yield {
                    "hidden": np.stack([b["hidden"] for b in batch]),
                    "tokens": (
                        np.stack([b["tokens"] for b in batch])
                        if "tokens" in batch[0]
                        else None
                    ),
                    "mask": np.stack([b["mask"] for b in batch]),
                }
                batch = []

        if batch:
            yield {
                "hidden": np.stack([b["hidden"] for b in batch]),
                "tokens": (
                    np.stack([b["tokens"] for b in batch])
                    if "tokens" in batch[0]
                    else None
                ),
                "mask": np.stack([b["mask"] for b in batch]),
            }

    def __len__(self) -> int:
        return self.total_examples // self.batch_size



## Trainer

In [ ]:
from flax.core import freeze, unfreeze
import json
import time
class Layer23Trainer:
    """
    Standalone trainer for TitansBlock on layer 23.

    Streams precomputed activations from HuggingFace Hub, runs them through
    the trainable TitansBlock, then through frozen Gemma layers 24-25 + head,
    and computes CrossEntropy loss on next-token prediction.

    Architecture:
        precomputed_hidden (B, L, 1152)
             ↓
        TitansBlock (layer 23) — ONLY this is trained
             ↓  (memory gate + retrieved + residual + MLP)
        Gemma layers 24–25 + final_norm + head  — FROZEN
             ↓
        logits → CrossEntropy loss
    """



    def __init__(
        self,
        merged_params: dict,
        optimizer,
        experimental_config: dict,
        activation_repo: str = "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
        activation_folder: str = "",
        local_activation_dir: Optional[str] = None,
        output_dir: str = "./checkpoints_layer23",
        batch_size: int = 4,
        seed: int = 42,
        hf_token: Optional[str] = None,
        cache_dir: Optional[str] = None,
    ):
        from gemma.gm.nn import _modules, _layers
        from gemma_titans import TitansBlock

        self.output_dir = output_dir
        self.batch_size = batch_size
        self.seed = seed
        os.makedirs(output_dir, exist_ok=True)

        # ---- 1. Use pre-built merged_params ----
        print("🔧 Using pre-built merged_params (Titans + Gemma)...")
        # merged_params is a flat dict: {layer_23:..., layer_24:..., final_norm:..., embedder:...}
        full_params = unfreeze(merged_params)
        param_keys = sorted(full_params.keys())
        print(f"   Top-level param keys: {param_keys}")

        # ---- 2. Build config from experimental_config ----
        config = build_titans_block_config(
            target_layer=TARGET_LAYER,
            experimental_config=experimental_config,
            every_k_schedule= 4
        )
        self.config = config

        # ---- 3. Create TitansBlock instance ----
        print("🏗️  Creating TitansBlock instance...")
        titans_block = TitansBlock(
            name=f'layer_{TARGET_LAYER}',
            **_build_block_kwargs(config, TARGET_LAYER),
            neural_mem_kwargs=config.neural_mem_kwargs,
            use_original_attn=False,  # Phase 3: pure memory, no attention
        )

        # ---- 4. Extract TitansBlock params from merged_params ----
        # merged_params already contains Phase 1/2 trained Titans weights
        # merged into the Gemma param tree at layer_{TARGET_LAYER}.
        print("📦 Extracting TitansBlock params from merged_params...")
        gemma_layer_key = f'layer_{TARGET_LAYER}'
        if gemma_layer_key not in full_params:
            raise KeyError(
                f"Layer '{gemma_layer_key}' not found in merged_params. "
                f"Available: {param_keys}"
            )
        self.titans_params = freeze(full_params[gemma_layer_key])


        # Print trainable param count
        param_count = sum(
            x.size for x in jax.tree_util.tree_leaves(self.titans_params)
        )
        print(f"   Trainable params: {param_count:,}")

        # ---- 6. Create frozen Gemma blocks ----
        print("❄️  Creating frozen Gemma blocks (layers 24, 25)...")
        block_24 = _modules.Block(
            name='layer_24',
            **_build_block_kwargs(config, 24),
        )
        block_25 = _modules.Block(
            name='layer_25',
            **_build_block_kwargs(config, 25),
        )

        # ---- 7. Extract frozen params ----
        for req_key in ['layer_24', 'layer_25', 'final_norm', 'embedder']:
            if req_key not in full_params:
                raise KeyError(
                    f"Required key '{req_key}' not found in merged_params."
                )

        frozen_params_24 = {'params': full_params['layer_24']}
        frozen_params_25 = {'params': full_params['layer_25']}

        final_norm = _layers.RMSNorm()
        frozen_final_norm_params = {'params': full_params['final_norm']}

        embedding_table = full_params['embedder']['input_embedding']
        print(f"   Embedding table: {embedding_table.shape}")
        print(
            f"   Frozen Block 24 attn_type: "
            f"{config.attention_types[24].name}"
        )
        print(
            f"   Frozen Block 25 attn_type: "
            f"{config.attention_types[25].name}"
        )

        # Free full checkpoint to save memory
        del full_params

        # ---- 8. Routing optimizer (M3 + Adam-atan2) ----
        print("⚙️  Setting up routing optimizer...")
        self.optimizer = optimizer
        self.opt_state = self.optimizer.init(self.titans_params)

        # ---- 9. Create train step ----
        print("🔧 Building train step (JIT compile on first call)...")
        self._train_step = make_train_step(
            titans_block=titans_block,
            block_24=block_24,
            block_25=block_25,
            final_norm_module=final_norm,
            frozen_params_24=frozen_params_24,
            frozen_params_25=frozen_params_25,
            frozen_final_norm_params=frozen_final_norm_params,
            embedding_table=embedding_table,
            optimizer=optimizer,
            neural_mem_kwargs=config.neural_mem_kwargs,
        )

        # ---- 10. Activation loader ----
        self.loader = HFActivationLoader(
            activation_repo=activation_repo,
            activation_folder=activation_folder,
            local_activation_dir=local_activation_dir,
            batch_size=batch_size,
            shuffle=True,
            seed=seed,
            hf_token=hf_token,
            cache_dir=cache_dir,
        )
        self.embed_dim = self.loader.embed_dim
        self.seq_len = self.loader.metadata.get("max_seq_len", 1024)

        print(f"   embed_dim={self.embed_dim}, seq_len={self.seq_len}")
        print("✅ Trainer initialized")

    def train(self, num_steps: int = 10000, eval_every: int = 500):
        """Main training loop — streams batches from HuggingFace."""
        import tensorflow as tf
        summary_writer = tf.summary.create_file_writer(self.output_dir)

        print(f"🚀 Starting training for {num_steps} steps...")
        print(
            f"   Data: {self.loader.activation_repo}"
            f"/{self.loader.activation_folder}"
        )

        step = 0
        t_start = time.time()
        loss_history = []

        # Внешний цикл для повторения датасета, если loader исчерпан
        while step < num_steps:
            for batch in self.loader:
                hidden = jnp.array(batch["hidden"])
                tokens_arr = batch["tokens"]
                mask = jnp.array(batch["mask"])

                if tokens_arr is None:
                    continue

                tokens = jnp.array(tokens_arr)

                self.titans_params, self.opt_state, loss, acc = self._train_step(
                    self.titans_params, self.opt_state, hidden, tokens, mask,
                )

                loss_val = float(loss)
                acc_val = float(acc)

                # TensorBoard logging
                if not np.isnan(loss_val):
                    with summary_writer.as_default():
                        tf.summary.scalar('train/loss', loss_val, step=step)
                        tf.summary.scalar('train/accuracy', acc_val, step=step)

                loss_history.append(loss_val)

                if step % 20 == 0:
                    elapsed = time.time() - t_start
                    steps_per_sec = (step + 1) / max(elapsed, 1e-6)
                    avg_loss = (
                        np.mean(loss_history[-20:]) if loss_history else 0.0
                    )
                    print(
                        f"  Step {step:6d} | loss={loss_val:.4f} | "
                        f"acc={acc_val:.4f} | "
                        f"avg={avg_loss:.4f} | "
                        f"{steps_per_sec:5.2f} steps/s | "
                        f"{elapsed:7.1f}s"
                    )

                if step > 0 and step % eval_every == 0:
                    self._save_checkpoint(step)

                step += 1
                if step >= num_steps:
                    break

        # Final checkpoint
        self._save_checkpoint(step)
        elapsed = time.time() - t_start
        print(f"✅ Training complete: {step} steps in {elapsed:.1f}s")

    def _save_checkpoint(self, step: int):
        """Save TitansBlock params checkpoint using Orbax."""
        import orbax
        from flax.training import orbax_utils

        ckpt_dir = os.path.join(self.output_dir, f"ckpt_{step}")
        ckpt_path = os.path.abspath(ckpt_dir)
        orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
        save_args = orbax_utils.save_args_from_target(self.titans_params)
        orbax_checkpointer.save(
            ckpt_path, self.titans_params, save_args=save_args,
            force=True,
        )
        print(f"   💾 Checkpoint saved: {ckpt_path}")


## Training step

In [ ]:
def make_train_step(
    titans_block,
    block_24,
    block_25,
    final_norm_module,
    frozen_params_24,
    frozen_params_25,
    frozen_final_norm_params,
    embedding_table,
    optimizer,
    neural_mem_kwargs,
):
    """
    Create a jitted training step.

    Architecture:
        hidden (B, L, 1152)
          → TitansBlock.apply(tp, ...)        [TRAINABLE]
          → Block_24.apply(frozen, ...)        [FROZEN]
          → Block_25.apply(frozen, ...)        [FROZEN]
          → RMSNorm.apply(frozen, ...)         [FROZEN]
          → dot(x, embedding_table.T) → logits [FROZEN]
          → CrossEntropy loss (shifted by 1)

    Only TitansBlock params receive gradients.
    Frozen params are closed-over constants (baked into XLA graph).
    Logit computation is @jax.checkpoint-ed to avoid materializing
    the full (B, L-1, 262144) tensor during backward.

    Returns:
        (new_titans_params, new_opt_state, loss_scalar, accuracy_scalar)
    """
    from titans import init_memory_state

    fp24 = frozen_params_24
    fp25 = frozen_params_25
    fnp = frozen_final_norm_params

    @jax.checkpoint
    def _compute_ce_loss_and_acc(x, targets, loss_mask):
        """Compute CE loss + accuracy from hidden states.

        Checkpointed so the huge (B, L-1, V) logit tensor is freed
        after forward and rematerialized on-demand during backward.
        Gemma3-1B has final_logit_softcap=None, so no softcap needed.
        """
        # x: (B, L, D) → logits: (B, L-1, V)
        # Numerical stability: cast to float32 and apply softcap
        x_f32 = x[:, :-1, :].astype(jnp.float32)
        logits = jnp.dot(x_f32, embedding_table.astype(jnp.float32).T)

        # Safe logit softcap (Gemma 3 style)
        softcap = 30.0
        logits = jnp.tanh(logits / softcap) * softcap

        ce = optax.softmax_cross_entropy_with_integer_labels(logits, targets)
        loss = (ce * loss_mask).sum() / jnp.maximum(loss_mask.sum(), 1.0)
        # Global NaN protection
        loss = jnp.nan_to_num(loss, nan=20.0)
        # Accuracy
        pred = jnp.argmax(logits, axis=-1)
        correct = (pred == targets).astype(jnp.float32)
        acc = (correct * loss_mask).sum() / jnp.maximum(loss_mask.sum(), 1.0)
        return loss, acc

    @jax.jit
    def train_step(titans_params, opt_state, hidden, tokens, mask):
        """
        Args:
            titans_params: TitansBlock parameters (trainable).
            opt_state: Optimizer state.
            hidden: Precomputed activations (B, L, 1152).
            tokens: Token IDs (B, L) — for CE target (shifted by 1).
            mask: Input mask (B, L) — 1 for real tokens, 0 for padding.
        Returns:
            (new_titans_params, new_opt_state, loss_scalar, accuracy_scalar)
        """
        B, L, D = hidden.shape

        # Create positions: (B, L)
        positions = jnp.broadcast_to(jnp.arange(L)[None, :], (B, L))

        # Create causal attention mask: (B, L, L)
        # Token i can attend to token j iff j <= i AND j is a real token.
        causal = jnp.tril(jnp.ones((L, L), dtype=jnp.bool_))
        attn_mask = causal[None, :, :] & mask[:, None, :].astype(jnp.bool_)

        # Fresh memory state for each batch (one-shot prefill)
        mem_state = init_memory_state(B, D, neural_mem_kwargs, dtype=hidden.dtype)

        def loss_fn(tp):
            # 1. TitansBlock (layer 23) — TRAINABLE
            cache_23 = {'memory_state': mem_state}
            _, x = titans_block.apply(
                {'params': tp}, hidden, positions, cache_23, attn_mask,
            )

            # 2. Block 24 — FROZEN (gradients still flow through for chain rule)
            _, x = block_24.apply(fp24, x, positions, None, attn_mask)

            # 3. Block 25 — FROZEN
            _, x = block_25.apply(fp25, x, positions, None, attn_mask)

            # 4. Final RMSNorm — FROZEN
            x = final_norm_module.apply(fnp, x)

            # 5. CE loss + accuracy (shifted by 1: predict next token)
            targets = tokens[:, 1:]
            loss_mask = mask[:, 1:].astype(jnp.float32)
            return _compute_ce_loss_and_acc(x, targets, loss_mask)

        (loss, acc), grads = jax.value_and_grad(loss_fn, has_aux=True)(titans_params)
        updates, new_opt_state = optimizer.update(grads, opt_state, titans_params)
        new_params = optax.apply_updates(titans_params, updates)

        return new_params, new_opt_state, loss, acc

    return train_step



In [ ]:
from routing_optimizer import make_routing_optimizer
_hld = experimental_config.get('huber_loss_delta', None)
huber_loss_delta = optax.constant_schedule(_hld) if _hld is not None else None
routing_optimizer = make_routing_optimizer(opt_params)

## 6. Loss и метрики

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext, kd

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss — основной сигнал Phase 2
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# Метрики

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class ValidRatio(kd_metrics.Metric):
    tokens: kontext.Key = "batch.tokens"
    pad_id: int = 0  # для Gemma обычно 0
    @flax.struct.dataclass
    class State(kd_metrics.base_state.AverageState):
        pass
    def get_state(self, *, tokens) -> State:
        # target positions for CLM
        tgt = tokens[:, 1:]  # [B, T]
        valid = (tgt != self.pad_id).astype(jnp.float32)
        # scalar per batch (0..1)
        valid_ratio = jnp.mean(valid)
        # AverageState ожидает значения; можно передать скаляр
        return self.State.from_values(values=valid_ratio)


@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return other

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, path_prefix=""):
            if hasattr(tree, 'items'):
                for key, val in tree.items():
                    current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                    if key == "memory_gate_proj":
                        # Handle Dense layer parameters (dict or single array)
                        leaves = jax.tree_util.tree_leaves(val)
                        all_params = jnp.concatenate([jnp.ravel(p) for p in leaves])
                        mean_val = jnp.mean(all_params)
                        openness = jax.nn.sigmoid(mean_val)
                        stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                        stats_dict[f"titans_gates/{current_path}_openness"] = openness
                        std_val = jnp.std(all_params)
                        stats_dict[f"titans_gates/{current_path}_raw_std"] = std_val

                    else:
                        find_gates(val, current_path)
        find_gates(params)
        return self.State(gate_metrics=freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class AdamLearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(opt_params['lr_adam'](step)))

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        # Состояние может быть пустым, так как данные мы берем напрямую из JAX на хосте
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    # Если словарь пустой, пропускаем устройство
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    # 1. Текущее использование памяти (если ключа нет, вернет 0)
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    # 2. Пиковое использование
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    # 3. Лимит и процент (только если 'limit' действительно существует)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            """Создает пустое начальное состояние."""
            return cls()
        def merge(self, other):
            """Объединяет состояния из разных батчей (здесь ничего не делаем)."""
            return self

    def get_state(self, **kwargs) -> State:
        # Просто возвращаем пустое состояние.
        # Нам не нужны данные из batch или модели для этой метрики.
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class Perplexity(kd.metrics.Metric):
    # Берет усредненный loss по батчу
    loss: kd.kontext.Key = "preds.layer_losses['lm_loss']"

    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        def compute(self):
            # Средний loss за все шаги -> возводим в экспоненту
            mean_loss = super().compute()
            return jnp.exp(mean_loss)
    def get_state(self,*, loss) -> State:
        return self.State.from_values(values=loss)

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LMAccuracy(kd.metrics.Metric):
    # Берет точность по батчу
    acc: kd.kontext.Key = "preds.layer_losses['lm_accuracy']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        pass
    def get_state(self,*, acc) -> State:
        return self.State.from_values(values=acc)


train_metrics = {
    "LM/accuracy": LMAccuracy(),
    # "LM/perplexity": Perplexity(),
    "LM/adam_lr": AdamLearningRateMetric(),
    # "LM/valid_ratio": ValidRatio(pad_id=0),
    # "memory_gate_proj": MemoryGateMetric(),
    "tpu_memory": TPUMemoryMetric()
}
train_summaries = {}
for layer in active_titans_layers:
    key = f"Gates_Dist_{layer}"
    train_summaries[key] = kd.summaries.HistogramSummary(tensor=f"params.layer_{layer}.memory_gate_proj.kernel")


## 3. Модель (training_phase=2)

In [ ]:
# !git pull

In [ ]:
# # import gemma_titans
# import gemma_titans
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config

## 7. Training


In [ ]:
# Проверка перед обучением
for _layer in loaded_titans_params:
    _gk = loaded_titans_params[_layer]['memory_gate_proj']['kernel']
    print(f"Layer {_layer} gate — Mean: {_gk.mean()}, Std: {_gk.std()}, Min: {_gk.min()}, Max: {_gk.max()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
workdir_name = f'titans_workdir_phase2_from{TARGET_LAYER}'

trainer = Layer23Trainer(
    output_dir=workdir_name,
    merged_params=merged_params,
    optimizer=routing_optimizer,
    experimental_config=experimental_config,
    activation_repo="veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    # local_activation_dir=args.local_activation_dir,
    batch_size=batch_size,
    cache_dir="/content/drive/Shareddrives/shared_veriga/jax_cache",
)

print(f"Trainer initialized. workdir: {workdir_name}")

## 8. TensorBoard

Если у вас уже запущено длительное обучение trainer.train() в ячейке, и нужно запустить заново Tensorboard, вам нужно воспользоваться Терминалом Colab:

terminal command available for runtime restart:
```
fuser -k 6006/tcp && tensorboard --logdir /content/Titans_jax/titans_workdir_phase2_from11/ --port 6006 &
```
после чего обновить Tensorboard


In [ ]:
%reload_ext tensorboard
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()



In [ ]:
!rm -rf /tmp/.tensorboard-info/*
!fuser -k 6006/tcp

In [ ]:
%tensorboard --logdir ./{workdir_name}/ --port=6006

## 9. Обучение

In [ ]:
trainer.train( )


new ver

In [ ]:
import jax

def print_tpu_mem_tpu_native():
    d = jax.devices()[0]
    ms = d.memory_stats() or {}

    used = ms.get("bytes_in_use", 0)
    peak = ms.get("peak_bytes_in_use", 0)

    limit = ms.get("bytes_limit", 0)
    reserved = ms.get("bytes_reserved", 0)
    reservable_limit = ms.get("bytes_reservable_limit", 0)
    largest_free = ms.get("largest_free_block_bytes", 0)

    print(f"Device: {d}")
    print(f"used_gb:               {used / 1e9:.2f}")
    print(f"peak_used_gb:          {peak / 1e9:.2f}")
    print(f"bytes_limit_gb:        {limit / 1e9:.2f}")
    print(f"bytes_reserved_gb:     {reserved / 1e9:.2f}")
    print(f"reservable_limit_gb:   {reservable_limit / 1e9:.2f}")
    print(f"largest_free_block_gb: {largest_free / 1e9:.2f}")

    if reservable_limit > 0:
        reservable_free = max(reservable_limit - reserved, 0)
        print(f"reservable_free_gb:    {reservable_free / 1e9:.2f}")

print_tpu_mem_tpu_native()

In [ ]:
import jax
import gc

# 1. DESTROY the Python references to the old TPU buffers FIRST
try:
    del state
    del aux
except NameError:
    pass

# 2. Clear JAX's internal live buffer cache
for device in jax.devices():
    if hasattr(device, 'live_arrays'): device.live_arrays().clear()
    if hasattr(device, 'live_buffers'): device.live_buffers().clear()
    if hasattr(device, 'default_memory_tracker'): device.default_memory_tracker().clear()

jax.clear_caches()

# 3. NOW run Python garbage collection
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")

print_tpu_mem_tpu_native()


In [ ]:

trainer = trainer.replace(num_train_steps=50000)

state, aux = trainer.train()

## 10. Сохранение весов

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    _, titans_params = titans_tree_utils.split_titans_params(state.params)
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved Titans weights to {save_path}")

new_weights_name = f"saved_titans_phase2_from_{TARGET_LAYER}_{total_steps}"
save_titans_weights(state, f"./{new_weights_name}")

# Upload to HuggingFace Hub
from google.colab import userdata

# 1. Загружаем чекпойнт (веса + метадата)
save_checkpoint_to_hf(
    save_dir=f"./{new_weights_name}",
    repo_id=HF_CKPT_REPO,
    phase=2,  # или 1 для TPUv6e-1
    warm_up=WARMUP,
    experimental_config=experimental_config,
    opt_params=opt_params,
    first_layer=TARGET_LAYER,
    total_steps=total_steps,
    token=userdata.get('HF_TOKEN'),
)

# 2. Обновляем «указатель» на последний эксперимент
save_last_metadata(
    repo_id=HF_CKPT_REPO,
    phase=2,  # или 1
    warm_up=WARMUP,
    first_layer=TARGET_LAYER,
    total_steps=total_steps,
    experimental_config=experimental_config,
    opt_params=opt_params,
    token=userdata.get('HF_TOKEN'),
)
print(f"✅ Phase 2 checkpoint + metadata uploaded to {HF_CKPT_REPO}")

## 11. Training Report

Сохраняем результаты обучения на HF, привязанные к гиперпараметрам:
- JSON-отчёт с loss-статистикой и гиперпараметрами
- PNG-график loss

In [ ]:
from hf_checkpoint import save_training_report, read_tensorboard_losses

# ── Читаем loss из TensorBoard ──
loss_history = read_tensorboard_losses(
    workdir=os.path.abspath(f'./{workdir_name}'),
    tag="lm_loss",
)
print(f"📊 Прочитано {len(loss_history)} точек loss из TensorBoard")

if loss_history:
    import numpy as np
    vals = [e["value"] for e in loss_history]
    print(f"   Loss: {vals[0]:.4f} → {vals[-1]:.4f}")
    print(f"   Last 500 avg: {np.mean(vals[-500:]):.4f}" if len(vals) >= 500 else "")

# ── Загружаем отчёт на HF ──
save_training_report(
    repo_id=HF_CKPT_REPO,
    phase=2,
    first_layer=TARGET_LAYER,
    total_steps=total_steps,
    loss_history=loss_history,
    extra_metrics={
        "batch_size": batch_size,
        "max_length": max_length,
        "active_titans_layers": list(active_titans_layers),
    },
    experimental_config=experimental_config,
    opt_params=opt_params,
    token=userdata.get('HF_TOKEN'),
)
print("✅ Training report uploaded!")